In [44]:
# =============================================================================
# FONCTION DE CHARGEMENT DU DATASET
#
# Charge :
# - X_train
# - X_test
# - y_train
# - y_test
#
# Transformation :
# - conversion de y_train et y_test en Series
# =============================================================================

import pandas as pd


def charger_dataset(
    chemin_X_train="../data/X_train_final.csv",
    chemin_X_test="../data/X_test_final.csv",
    chemin_y_train="../data/y_train.csv",
    chemin_y_test="../data/y_test.csv"
):

    X_train = pd.read_csv(chemin_X_train)
    X_test = pd.read_csv(chemin_X_test)

    y_train = pd.read_csv(chemin_y_train)
    y_test = pd.read_csv(chemin_y_test)


    # Transformation en Series pour sklearn
    y_train = y_train.squeeze()
    y_test = y_test.squeeze()


    print("Dataset chargé")
    print("------------------------------")
    print("X_train :", X_train.shape)
    print("X_test  :", X_test.shape)
    print("y_train :", y_train.shape)
    print("y_test  :", y_test.shape)


    return X_train, X_test, y_train, y_test

In [45]:
# =============================================================================
# FONCTION DE PREPARATION DES 3 DATASETS POUR COMPARAISON
#
# Génère :
# 1) marques conservées + variables homologation supprimées
# 2) marques supprimées + variables homologation supprimées
# 3) toutes les variables conservées (avec homologation)
#
# =============================================================================

def preparer_datasets_comparaison(
    X_train,
    X_test,
    y_train,
    y_test
):

    X_train_base = X_train.copy()
    X_test_base = X_test.copy()


    # Variables pouvant créer une fuite de données
    cols_homologation = [
        "co_typ_1",
        "conso_mixte",
        "nox",
        "ptcl"
    ]


    # Variables marques
    cols_marques = [
        col for col in X_train.columns
        if col.startswith("lib_mrq_")
    ]


    # -------------------------------------------------------------------------
    # 1 - Marques conservées / homologation supprimée
    # -------------------------------------------------------------------------

    X_train_marques = X_train_base.drop(
        columns=cols_homologation,
        errors="ignore"
    )

    X_test_marques = X_test_base.drop(
        columns=cols_homologation,
        errors="ignore"
    )


    # -------------------------------------------------------------------------
    # 2 - Marques supprimées / homologation supprimée
    # -------------------------------------------------------------------------

    cols_suppression = (
        cols_homologation
        +
        cols_marques
    )


    X_train_sans_marques = X_train_base.drop(
        columns=cols_suppression,
        errors="ignore"
    )

    X_test_sans_marques = X_test_base.drop(
        columns=cols_suppression,
        errors="ignore"
    )


    # -------------------------------------------------------------------------
    # 3 - Toutes variables conservées
    # -------------------------------------------------------------------------

    X_train_avec_homologation = X_train_base.copy()
    X_test_avec_homologation = X_test_base.copy()



    datasets = {

        "marques_sans_homologation":
        (
            X_train_marques,
            X_test_marques,
            y_train,
            y_test
        ),


        "sans_marques_sans_homologation":
        (
            X_train_sans_marques,
            X_test_sans_marques,
            y_train,
            y_test
        ),


        "avec_variables_homologation":
        (
            X_train_avec_homologation,
            X_test_avec_homologation,
            y_train,
            y_test
        )
    }


    # Contrôle

    for nom, (Xtr, Xte, _, _) in datasets.items():

        print("------------------------------")
        print(nom)
        print("X_train :", Xtr.shape)
        print("X_test  :", Xte.shape)


        assert list(Xtr.columns) == list(Xte.columns)


    return datasets

    for nom, (Xtr, Xte) in datasets.items():

        print("--------------------------------")
        print(nom)
        print("--------------------------------")
        print("X_train :", Xtr.shape)
        print("X_test  :", Xte.shape)


        # Vérification train/test identiques
        assert list(Xtr.columns) == list(Xte.columns), \
            "Erreur : colonnes train/test différentes"


    return {

        "marques_sans_homologation": 
            (X_train_marques, X_test_marques, y_train, y_test),

        "sans_marques_sans_homologation":
            (X_train_sans_marques, X_test_sans_marques, y_train, y_test),

        "avec_homologation":
            (X_train_homologation, X_test_homologation, y_train, y_test)

    }



In [ ]:
# =============================================================================
# COMPARAISON DES MODELES AVEC VALIDATION CROISEE
# =============================================================================


import pandas as pd

from sklearn.model_selection import (
    KFold,
    cross_validate,
    GridSearchCV
)

from sklearn.linear_model import (
    LinearRegression,
    RidgeCV,
    LassoCV
)

from sklearn.tree import DecisionTreeRegressor



def comparer_modeles_regression(
    X_train,
    y_train,
    nom_test
):


    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )


    # -------------------------------------------------------------------------
    # Ridge
    # -------------------------------------------------------------------------

    ridge = RidgeCV(
        alphas=[
            0.001,
            0.01,
            0.1,
            0.3,
            0.7,
            1,
            10,
            50,
            100
        ],
        cv=cv,
        scoring="r2"
    )


    ridge.fit(
        X_train,
        y_train
    )


    print(
        "Meilleur alpha Ridge :",
        ridge.alpha_
    )


    # -------------------------------------------------------------------------
    # Lasso
    # -------------------------------------------------------------------------

    lasso = LassoCV(
        alphas=[
            0.001,
            0.01,
            0.1,
            0.3,
            0.7,
            1,
            10
        ],
        cv=cv,
        max_iter=10000,
        random_state=42
    )


    lasso.fit(
        X_train,
        y_train
    )


    print(
        "Meilleur alpha Lasso :",
        lasso.alpha_
    )


    # -------------------------------------------------------------------------
    # Arbre
    # -------------------------------------------------------------------------

    tree = DecisionTreeRegressor(
        random_state=42
    )


    param_grid = {

        "max_depth":[5,10,15,20,None],

        "min_samples_split":[2,5,10,20],

        "min_samples_leaf":[1,2,5,10]

    }


    tree_grid = GridSearchCV(
        tree,
        param_grid,
        cv=cv,
        scoring="r2",
        n_jobs=1
    )


    tree_grid.fit(
        X_train,
        y_train
    )


    best_tree = tree_grid.best_estimator_


    print(
        "Meilleurs paramètres arbre :",
        tree_grid.best_params_
    )



    models = {

        "Régression linéaire":
            LinearRegression(),

        "Ridge optimisé":
            ridge,

        "Lasso optimisé":
            lasso,

        "Arbre décision optimisé":
            best_tree

    }



    resultats = []


    for nom_modele, modele in models.items():

        scores = cross_validate(

            modele,

            X_train,

            y_train,

            cv=cv,

            scoring={

                "R2":"r2",

                "RMSE":
                "neg_root_mean_squared_error",

                "MAE":
                "neg_mean_absolute_error",

                "MAPE":
                "neg_mean_absolute_percentage_error"

            },

            n_jobs=1
        )


        resultats.append({

            "Modèle":
            nom_modele,

            "R² moyen":
            scores["test_R2"].mean(),

            "R² écart-type":
            scores["test_R2"].std(),

            "RMSE moyen":
            -scores["test_RMSE"].mean(),

            "MAE moyen":
            -scores["test_MAE"].mean(),

            "MAPE moyen (%)":
            -scores["test_MAPE"].mean()*100

        })


    results_df = pd.DataFrame(resultats)


    results_df = results_df.sort_values(
        by="R² moyen",
        ascending=False
    )


    results_df = results_df.round(4)


    display(results_df)



    fichier = (
        "Comparaison_modeles_"
        + nom_test
        + ".xlsx"
    )


    results_df.to_excel(
        fichier,
        index=False
    )


    print(
        "Export terminé :",
        fichier
    )


    return results_df

In [47]:
#Appeller la fonction comparer_modeles_regression pour chaque dataset testé:
# =============================================================================
# APPEL DE LA FONCTION comparer_modeles_regression
# POUR CHAQUE DATASET TESTE
# =============================================================================


X_train, X_test, y_train, y_test = charger_dataset()


datasets = preparer_datasets_comparaison(
    X_train,
    X_test,
    y_train,
    y_test
)


resultats_comparaison = {}


for nom_dataset, (Xtr, Xte, ytr, yte) in datasets.items():

    print("\n")
    print("="*80)
    print("TEST :", nom_dataset)
    print("="*80)


    resultats_comparaison[nom_dataset] = comparer_modeles_regression(
        Xtr,
        ytr,
        nom_dataset
    )

Dataset chargé
------------------------------
X_train : (44008, 91)
X_test  : (11002, 91)
y_train : (44008,)
y_test  : (11002,)
------------------------------
marques_sans_homologation
X_train : (44008, 87)
X_test  : (11002, 87)
------------------------------
sans_marques_sans_homologation
X_train : (44008, 44)
X_test  : (11002, 44)
------------------------------
avec_variables_homologation
X_train : (44008, 91)
X_test  : (11002, 91)


TEST : marques_sans_homologation
Meilleur alpha Ridge : 0.1
Meilleur alpha Lasso : 0.001


Python(46393) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(46394) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(46395) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(46396) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(46397) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(46398) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(46399) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(46400) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(46401) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(46402) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Meilleurs paramètres arbre : {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}


,Modèle,R² moyen,R² écart-type,RMSE moyen,MAE moyen,MAPE moyen (%)
3,Arbre décision optimisé,0.9505,0.0019,7.5351,5.4434,2.8281
0,Régression linéaire,0.8546,0.0058,12.9100,9.8871,5.0807
1,Ridge optimisé,0.8546,0.0057,12.9121,9.8894,5.0803
2,Lasso optimisé,0.8543,0.0054,12.9236,9.9009,5.0833


Export terminé : Comparaison_modeles_marques_sans_homologation.xlsx


TEST : sans_marques_sans_homologation
Meilleur alpha Ridge : 0.1
Meilleur alpha Lasso : 0.001
Meilleurs paramètres arbre : {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}


,Modèle,R² moyen,R² écart-type,RMSE moyen,MAE moyen,MAPE moyen (%)
3,Arbre décision optimisé,0.9472,0.0019,7.7773,5.5170,2.8643
1,Ridge optimisé,0.8285,0.0061,14.0193,10.4753,5.3429
0,Régression linéaire,0.8285,0.0062,14.0205,10.4759,5.3452
2,Lasso optimisé,0.8283,0.0059,14.0275,10.4823,5.3436


Export terminé : Comparaison_modeles_sans_marques_sans_homologation.xlsx


TEST : avec_variables_homologation
Meilleur alpha Ridge : 0.001
Meilleur alpha Lasso : 0.001
Meilleurs paramètres arbre : {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2}


,Modèle,R² moyen,R² écart-type,RMSE moyen,MAE moyen,MAPE moyen (%)
3,Arbre décision optimisé,0.9989,0.0006,1.0560,0.0894,0.0723
0,Régression linéaire,0.9971,0.0003,1.8266,1.1099,0.6116
1,Ridge optimisé,0.9971,0.0003,1.8267,1.1106,0.6118
2,Lasso optimisé,0.9970,0.0003,1.8464,1.1392,0.6245


Export terminé : Comparaison_modeles_avec_variables_homologation.xlsx
